# FAIR Organelle Segmentation pipeline

In [ ]:
import sys
sys.path.append('..')

# Relies on https://github.com/volume-em/empanada-napari.git@inf_pipeline_dev
from empanada_napari._slice_inference import SliceInferenceWidget
import glob
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
from napari.components import ViewerModel
import numpy as np
import ome_zarr
from random import random
from skimage import measure
from skimage.transform import resize

from src.fair_segmentation.image_util import *

## Load source data

In [ ]:
source_path = 'D:/slides/AMC_EM/**/*.tif'

filenames = glob.glob(source_path)
filenames


## Process data

In [ ]:
downscale = 2

datas = []
for filename in filenames:
    data, metadata, pixel_size = extract_tiff_olympus(filename)
    new_shape = list(data.shape)
    new_shape[0] //= downscale
    new_shape[1] //= downscale
    data = resize(data, new_shape)
    datas.append(float2int_image(norm_image_quantiles(data)))

## Select data

In [ ]:
%matplotlib inline

data = datas[0]

plt.imshow(data, cmap='gray')

## Run model

In [ ]:
viewer = ViewerModel()
image_layer = viewer.add_image(data)

inference_config = SliceInferenceWidget(viewer=viewer,
                                        image_layer=image_layer,
                                        model_config='MitoNet_v1',
                                        use_quantized=True,
                                        use_gpu=False)
seg, axis, plane, y, x = inference_config.config_and_run_inference(use_thread=False)

## Show output

In [ ]:
%matplotlib inline
seg[seg >= 10000] -= 10000
maxval = np.max(seg)

regions = measure.regionprops(seg)
for props in regions:
    y0, x0 = props.centroid
    print(f"{props.label}: {props.area}")

## Draw output

In [ ]:
colors = [(0,0,0)] + [(random(),random(),random()) for _ in range(maxval)]
new_map = LinearSegmentedColormap.from_list('random', colors, N=maxval+1)
plt.imshow(data, cmap='gray')
plt.imshow(seg, cmap=new_map, alpha=0.75)